In [1]:
import pandas as pd
from numpy.random import default_rng
from pathlib import Path

def select_random_rows_excel(input_path: str, n_rows: int, output_csv: str, seed=27):
    """
    Randomly select `n_rows` unique rows from an Excel sheet and save them to `output_csv`.

    Parameters
    ----------
    input_path : str
        Path to the Excel file (.xlsx, .xls).
    n_rows : int
        Number of rows to select, e.g., 500.
    output_csv : str
        Path to save the selected rows as CSV.
    seed : int | None, optional
        Random seed for reproducibility.

    Returns
    -------
    pandas.DataFrame
        The selected rows.
    """
    input_path = Path(input_path)
    if not input_path.exists():
        raise FileNotFoundError(f"File not found: {input_path}")

    # Read Excel
    df = pd.read_csv(input_path)

    if n_rows <= 0:
        raise ValueError("n_rows must be positive.")
    if n_rows > len(df):
        raise ValueError(f"Requested {n_rows} rows, but file has only {len(df)} rows.")

    rng = default_rng(seed)

    # Iterative selection without replacement (matches your step-by-step description)
    remaining_idx = df.index.to_numpy()
    selected_idx = []
    for _ in range(n_rows):
        pick = rng.choice(remaining_idx)
        selected_idx.append(pick)
        remaining_idx = remaining_idx[remaining_idx != pick]  # exclude picked index

    out_df = df.loc[selected_idx].reset_index(drop=True)
    out_df.to_csv(output_csv, index=False)
    return out_df

# Example:
# select_random_rows_excel("data.xlsx", 500, "sampled_rows.csv")


In [3]:

# Example:
select_random_rows_excel("Alabama_Test/polygon_areas_filtered.csv", 50, "Alabama_Test/01_filtered_polygon_areas_ch_1_test.csv", seed=7)
select_random_rows_excel("Alabama_Test/polygon_areas_filtered.csv", 50, "Alabama_Test/02_filtered_polygon_areas_ch_1_test.csv", seed=17)
select_random_rows_excel("Alabama_Test/polygon_areas_filtered.csv", 50, "Alabama_Test/03_filtered_polygon_areas_ch_1_test.csv", seed=27)
select_random_rows_excel("Alabama_Test/polygon_areas_filtered.csv", 50, "Alabama_Test/04_filtered_polygon_areas_ch_1_test.csv", seed=37)
select_random_rows_excel("Alabama_Test/polygon_areas_filtered.csv", 50, "Alabama_Test/05_filtered_polygon_areas_ch_1_test.csv", seed=47)

,latitude,longitude,class,confidence,tile_name,area_geo,area_gis,length,width,area_sqft,centroid_x,centroid_y,mask_points_gis
0,3.492159e+06,620757.870905,1,0.876156,m_3108527_sw_16_030_20211109.tif,2.88,43756.060793,612.227339,78.002953,43756.103137,620757.284426,3.492159e+06,"[[620845.05, 3492174.75], [620845.3500000001, ..."
1,3.789266e+06,518873.652283,1,0.858107,m_3408650_ne_16_030_20211126.tif,0.36,21273.771960,399.901585,57.842992,21273.792547,518873.963593,3.789267e+06,"[[518865.15, 3789274.9499999997], [518865.75, ..."
2,3.862606e+06,623983.883752,1,0.943216,m_3408503_se_16_030_20211027.tif,6.48,33627.769087,525.003670,72.812234,33627.801631,623983.383478,3.862607e+06,"[[623984.2500000002, 3862622.55], [623984.55, ..."
3,3.787473e+06,537210.654895,1,0.886034,m_3408652_nw_16_030_20211126.tif,1.17,27111.949861,533.004020,57.161496,27111.976099,537210.731959,3.787476e+06,"[[537217.9500000001, 3787556.85], [537218.25, ..."
4,3.756768e+06,505565.222021,1,0.871736,m_3308601_nw_16_030_20211126.tif,11.52,19657.895292,405.396269,52.181010,19657.914316,505564.564038,3.756769e+06,"[[505516.05, 3756805.0500000003], [505516.6499..."
5,3.739023e+06,419031.208447,0,0.723214,m_3308709_se_16_030_20211108.tif,2.07,36942.108446,620.078760,71.850396,36942.144197,419032.560389,3.739023e+06,"[[419126.55, 3739027.35], [419126.85000000003,..."
6,3.482445e+06,646791.778711,1,0.918470,m_3108537_nw_16_030_20211114.tif,14.04,26018.714358,515.779688,57.545532,26018.739537,646792.881793,3.482447e+06,"[[646816.0499999999, 3482521.0500000003], [646..."
7,3.486408e+06,647052.022925,1,0.887594,m_3108529_sw_16_030_20211114.tif,1.08,27881.138155,507.566404,61.415013,27881.165137,647053.709552,3.486409e+06,"[[646979.55, 3486419.25], [646983.45, 3486419...."
8,3.495161e+06,619827.182245,1,0.924146,m_3108527_nw_16_030_20211109.tif,15.57,27007.324753,532.651647,57.703391,27007.350890,619826.770478,3.495163e+06,"[[619793.25, 3495235.65], [619795.0499999999, ..."
9,3.790727e+06,426495.967969,1,0.918201,m_3408742_se_16_030_20211106.tif,11.97,29109.514424,506.783509,63.754842,29109.542594,426495.894828,3.790728e+06,"[[426428.55, 3790759.0500000003], [426428.8500..."


In [10]:
# Convert the uploaded CSV to a Google Earth KML (WGS84) using polygons where available.
import ast
import csv
from pathlib import Path
import xml.etree.ElementTree as ET
from pyproj import CRS, Transformer

def csv_to_kml_polygons(
    input_csv: str,
    output_kml: str,
    source_epsg=26916,                              # UTM 16N for your data
    name_col: str = "tile_name",
    polygon_col: str = "mask_points_gis",
    centroid_cols: tuple[str, str] = ("centroid_x", "centroid_y"),
    fallback_xy_cols: tuple[str, str] = ("longitude", "latitude"),
    desc_cols: tuple[str, ...] = ("class", "confidence", "area_geo", "area_gis", "area_sqft", "length", "width"),
    include_extended_data: bool = True,
):
    # Build transformer to WGS84
    src = CRS.from_epsg(int(str(source_epsg).replace("EPSG:", "")))
    dst = CRS.from_epsg(4326)
    tfm = Transformer.from_crs(src, dst, always_xy=True)

    p = Path(input_csv)
    if not p.exists():
        raise FileNotFoundError(input_csv)

    def ring_to_kml_coords(xy_list):
        coords = []
        for xy in xy_list:
            if not isinstance(xy, (list, tuple)) or len(xy) < 2:
                continue
            lon, lat = tfm.transform(float(xy[0]), float(xy[1]))
            coords.append(f"{lon},{lat},0")
        if len(coords) >= 3 and coords[0] != coords[-1]:
            coords.append(coords[0])
        return coords

    kml_ns = "http://www.opengis.net/kml/2.2"
    kml = ET.Element("kml", {"xmlns": kml_ns})
    doc = ET.SubElement(kml, "Document")

    with p.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        fields = reader.fieldnames or []

        for i, row in enumerate(reader, start=1):
            pm = ET.SubElement(doc, "Placemark")

            # Name
            nm = row.get(name_col, "").strip() if name_col in fields else f"Feature {i}"
            ET.SubElement(pm, "name").text = nm

            # Description
            if desc_cols:
                parts = [f"{c}: {row[c]}" for c in desc_cols if c in fields and row.get(c, "") != ""]
                if parts:
                    ET.SubElement(pm, "description").text = "\n".join(parts)

            # ExtendedData
            if include_extended_data:
                ext = ET.SubElement(pm, "ExtendedData")
                for k, v in row.items():
                    if k == polygon_col:
                        continue
                    d = ET.SubElement(ext, "Data", name=k)
                    ET.SubElement(d, "value").text = "" if v is None else str(v)

            # Geometry
            wrote_geom = False
            s = row.get(polygon_col, "")
            if isinstance(s, str) and s.strip().startswith("["):
                try:
                    poly_obj = ast.literal_eval(s)
                    if poly_obj and isinstance(poly_obj[0], (list, tuple)) and isinstance(poly_obj[0][0], (int, float)):
                        coords = ring_to_kml_coords(poly_obj)
                        if len(coords) >= 4:
                            poly = ET.SubElement(pm, "Polygon")
                            outer = ET.SubElement(poly, "outerBoundaryIs")
                            ring = ET.SubElement(outer, "LinearRing")
                            ET.SubElement(ring, "coordinates").text = " ".join(coords)
                            wrote_geom = True
                    elif poly_obj and isinstance(poly_obj[0], (list, tuple)) and isinstance(poly_obj[0][0], (list, tuple)):
                        mg = ET.SubElement(pm, "MultiGeometry")
                        any_written = False
                        for ring_like in poly_obj:
                            coords = ring_to_kml_coords(ring_like)
                            if len(coords) >= 4:
                                poly = ET.SubElement(mg, "Polygon")
                                outer = ET.SubElement(poly, "outerBoundaryIs")
                                ring = ET.SubElement(outer, "LinearRing")
                                ET.SubElement(ring, "coordinates").text = " ".join(coords)
                                any_written = True
                        wrote_geom = any_written
                except Exception:
                    wrote_geom = False

            if not wrote_geom:
                if all(c in fields and row.get(c, "") != "" for c in centroid_cols):
                    x = float(row[centroid_cols[0]]); y = float(row[centroid_cols[1]])
                else:
                    x = float(row[fallback_xy_cols[0]]); y = float(row[fallback_xy_cols[1]])
                lon, lat = tfm.transform(x, y)
                pt = ET.SubElement(pm, "Point")
                ET.SubElement(pt, "coordinates").text = f"{lon},{lat},0"

    Path(output_kml).parent.mkdir(parents=True, exist_ok=True)
    ET.ElementTree(kml).write(output_kml, encoding="utf-8", xml_declaration=True)

# Convert the uploaded file
in_csv = "Alabama_Test/Chapter_1_test/05_filtered_polygon_areas_ch_1_test.csv"
out_kml = "Alabama_Test/Chapter_1_test/05_filtered_polygon_areas_ch_1_test.kml"
csv_to_kml_polygons(in_csv, out_kml, source_epsg=26916)
print("Wrote:", out_kml)


Wrote: Alabama_Test/Chapter_1_test/05_filtered_polygon_areas_ch_1_test.kml
